# GeoMM-Bench Experiment

All CLIP approaches use one backbone, `openai/clip-vit-base-patch32`; Grounding DINO and BLIP-2 are separate models. This notebook runs the same code as `run_geomm_bench.py` and writes the same results file, including a `backbone_sensitivity` block.

In the pilot, only text classifies reasonably (~0.73 macro-F1); the vision, grounding, VQA and added-modality approaches do poorly, and adding Full Wave Sonic makes the fusion worse. Vision-CLIP also varies a lot by backbone (around 0.09 on OpenAI CLIP vs 0.62 on LAION), which is why the sensitivity block reports both.

## 1. Setup

In [ ]:
# Dependencies (uncomment on first run)
# %pip install -q torch transformers Pillow numpy matplotlib pdf2image
# %pip install -q open-clip-torch   # optional, for the backbone-sensitivity check
# macOS: brew install poppler   |   Linux: apt-get install poppler-utils

In [ ]:
import os, sys, json
from collections import Counter
sys.path.insert(0, os.path.abspath('.'))
from geomm_bench import baselines as B
print('backbone:', B.CLIP_MODEL_NAME)

## 2. Ground truth (11 labelled intervals)

In [ ]:
GT = json.load(open('data/ground_truth.json'))['intervals']
print(len(GT), 'intervals;', dict(Counter(g['lithology'] for g in GT)))

## 3. Source PDFs (optional — needed for the image approaches)

Set these to your operator rasters; leave them missing to run text-only only. The PDFs are not redistributed (see `DATASHEET.md`).

In [ ]:
LOGS_PDF = 'vilkyciai22_logs500.pdf'    # set to your path
FWS_PDF  = 'vilkyciai22_fws_im_dt.pdf'  # set to your path
LOGS_PDF = LOGS_PDF if os.path.exists(LOGS_PDF) else None
FWS_PDF  = FWS_PDF  if os.path.exists(FWS_PDF)  else None
print('logs:', LOGS_PDF, '| fws:', FWS_PDF)

## 4. Run the benchmark (all approaches + sensitivity)

In [ ]:
import run_geomm_bench as R

log_pages = R._load_pages(LOGS_PDF) if LOGS_PDF else None
fws_pages = R._load_pages(FWS_PDF) if FWS_PDF else None
doc = R.build_doc(GT, log_pages, fws_pages)   # primary results + backbone_sensitivity

OUT_JSON = 'results/geomm_bench_results.json'
os.makedirs('results', exist_ok=True)
json.dump(doc, open(OUT_JSON, 'w'), indent=2)
for a in R.ALL_APPROACHES:
    r = doc['results'][a]
    print(f"{a:24s} F1={r['macro_f1']} acc={r['accuracy']}")
print('sensitivity (text_only):',
      {bk: blk['results']['text_only']['macro_f1'] for bk, blk in doc['backbone_sensitivity'].items()})

## 5. Figure (plotted from the results file)

In [ ]:
import subprocess
subprocess.run([sys.executable, 'scripts/make_results_figure.py',
                '--results', OUT_JSON,
                '--out', 'images/Figure2_Results_Comparison.png'])

## 6. Conclusion

Only text-only classification works on this pilot (~0.73 macro-F1). The image-based approaches — vision, grounding, VQA, fusion, and the added-modality variants — all do poorly, and adding the Full Wave Sonic display lowers the fusion score.

Vision-CLIP also depends heavily on the CLIP backbone (about 0.09 on OpenAI CLIP vs 0.62 on LAION), while text-only is about the same on both. So the result is better read as off-the-shelf vision being unreliable here than as vision simply failing. The limit seems to be the visual representation rather than the amount of input data.